In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'CMacedonia'

In [4]:
YEAR = 2023
MONTH = 'May'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.46197,40.60446,2023-05-01,κεντρικης μακεδονιας,αλεξανδρειας,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,478.8,38293.0,86.8,46.40326,0.311086,0.012307,-0.309329,-0.012307,0.307588,0.007922,-0.309126,-0.007922,0.075574,0.063291,0.052298,0.063291,20.233636,28.522727,11.944545,11.510466,1.708510,11.778876,0.439805,21.186067,5.575849,24.421964,8.937268,4.466450,8.214727,59.913172,18483.883785,1629.598106,4,120.472193,7.866005,180.191380,0.0,1.400071,31,99,36,99.0,30,99,12,12,1,6,7,2,0,111,0,0
1,22.07722,41.01522,2023-05-01,κεντρικης μακεδονιας,αλμωπιας,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,985.8,24924.0,28.0,46.57952,0.533142,0.127337,-0.456135,-0.127337,0.533186,0.127902,-0.456785,-0.127902,0.060746,0.059026,0.036408,0.059026,12.568704,17.027037,8.110370,9.238694,1.920162,9.573419,-0.371554,15.120878,3.834306,16.704092,6.474975,4.031016,18.822617,139.845076,25701.766035,44.089931,2,194.572797,144.613300,181.767317,0.0,113.550432,31,93,36,94.0,30,93,12,12,3,5,8,2,0,162,0,0
2,22.89198,40.65603,2023-05-01,κεντρικης μακεδονιας,αμπελοκηπων μενεμενης,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,9.8,49674.0,5319.1,46.65786,0.157606,0.030411,-0.169270,-0.030411,0.198656,0.031271,-0.205202,-0.031271,0.092685,0.059763,0.069722,0.059763,19.870000,27.870000,11.870000,12.381429,3.596667,11.870000,2.525385,18.760769,5.121429,23.898000,11.870000,8.539829,24.316921,129.586469,1013.723139,1259.926671,2,166.963790,4.655965,180.457193,0.0,24.343007,32,97,9,99.0,30,97,13,13,10,8,9,2,0,156,0,0
3,23.95900,40.91196,2023-05-01,κεντρικης μακεδονιας,αμφιπολης,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,411.8,7169.0,22.3,47.41120,0.484106,0.193471,-0.423805,-0.193471,0.434420,0.188544,-0.373615,-0.188544,0.070452,0.066047,0.055693,0.066047,15.578000,21.420667,9.735333,10.655161,3.113550,9.895287,0.416503,16.682835,4.861432,20.539981,7.233860,14.148982,39.067233,79.507399,14413.246350,196.335052,5,220.810138,250.710581,185.823044,0.0,25.464784,31,88,30,88.0,30,88,10,10,1,6,6,2,0,78,0,0
4,23.69747,40.49593,2023-05-01,κεντρικης μακεδονιας,αριστοτελη,1,5,18,2023,0.201299,0.97953,0.5,-0.866025,0.845596,-0.533823,747.0,16994.0,24.5,46.92004,0.575123,0.233051,-0.460720,-0.233051,0.572054,0.261183,-0.452363,-0.261183,0.062169,0.051572,0.047463,0.051572,13.910556,18.508889,9.312222,10.968774,3.770478,9.453781,2.661346,15.362634,4.844974,17.259672,7.381085,12.936044,87.667840,152.517369,11468.297401,1749.385768,22,273.587292,768.043224,181.562024,0.0,1.033661,14,99,10,99.0,10,99,4,4,6,4,4,2,0,39,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

scaler = MinMaxScaler()
imputer = KNNImputer()

X_test = scaler.fit_transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,23.46965,40.01303,κασσανδρας,1,5,2023,0.995856
1,22.90326,40.67609,κορδελιου ευοσμου,1,5,2023,0.992404
2,23.87955,40.08302,σιθωνιας,1,5,2023,0.991180
3,23.08410,40.49006,θερμης,1,5,2023,0.990553
4,23.19403,40.32484,νεας προποντιδας,1,5,2023,0.988482
5,23.27201,41.13057,ηρακλειας,1,5,2023,0.986334
6,22.17237,40.79674,σκυδρας,1,5,2023,0.969100
7,23.42474,40.40349,πολυγυρου,1,5,2023,0.965371
8,22.90881,40.77096,ωραιοκαστρου,1,5,2023,0.964417
9,23.67524,41.04497,εμμανουηλ παππα,1,5,2023,0.963102


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0648252687608289, 0.3787123377107355, 0.6803072708219687, 0.838221147727126, 0.935474790938512, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,23.46965,40.01303,κασσανδρας,1,5,2023,0.995856,5
1,22.90326,40.67609,κορδελιου ευοσμου,1,5,2023,0.992404,5
2,23.87955,40.08302,σιθωνιας,1,5,2023,0.991180,5
3,23.08410,40.49006,θερμης,1,5,2023,0.990553,5
4,23.19403,40.32484,νεας προποντιδας,1,5,2023,0.988482,5
5,23.27201,41.13057,ηρακλειας,1,5,2023,0.986334,5
6,22.17237,40.79674,σκυδρας,1,5,2023,0.969100,5
7,23.42474,40.40349,πολυγυρου,1,5,2023,0.965371,5
8,22.90881,40.77096,ωραιοκαστρου,1,5,2023,0.964417,5
9,23.67524,41.04497,εμμανουηλ παππα,1,5,2023,0.963102,5


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}_(new).csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results